# V00_export: migration flows, nodes and per-person paths (layer 4)
Tasks `tasks/T-005-v00-export-migration.md` (brief B-016) and `tasks/T-013-v00-decade-hubs.md` (brief B-025: decade per leg, people-weighted state hubs; D-025). SQL + pandas only, no ML/LLM. Reads `data/processed/` (and the Newberry state shapes in `data/raw/gis/`, read-only, for the hub check), writes only `web/data/`.

| Step | Purpose |
|---|---|
| 0.1 | Stops: `residences` + `households.served_state` as a `served` stop, ordered per NAID × person |
| 0.2 | Legs: consecutive stops where the place changes; `decade` per leg (dated year, else application year); Act cohort from `application_year` |
| 0.3 | `flows` (from × to × decade, state and county rows) and `nodes` (place × decade, people left / arrived; people-weighted hub per state) |
| 0.4 | Counts: legs per kind pair, people per cohort, files with ≥2 stops |
| 0.5 | Assertions (totals = T-005, hubs inside their state), write `web/data/flows.json`, `nodes.json`, `paths.json` |

**Sources:** `data/processed/residences.parquet` (111,317 rows), `households.parquet` (78,745 files), `places.parquet` (1,919 places; A05–A08, 2026-11-22).

**Rules (T-005, PLAN §3; T-013, D-025):** all claim types count (S, W, R, BLWT); each leg counts once; legs share the decade timeline 1770–1840 (D-025 reverses the old "no decade bins" rule); the Act cohort stays as a per-row tally.

**Caveats** (`docs/DATA.md`): residences come from regex over declarations (A07: 83% row-weighted strict). Most stops are undated; interim stops (`removed_to`, `lived_since`, `married`) are ordered by date, then page, so their order is a best guess when undated. In W files the widow's chain is her own (`person = widow`), not the veteran's. `served` is the state of the line he served in, not always where he lived. MA → ME is partly administrative (Maine ⊂ MA until 1820).

In [1]:
import sys, json
from pathlib import Path
PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path: sys.path.insert(0, str(PIPELINE_DIR))
import config
import duckdb, pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 120)
con = duckdb.connect()
for t in ["residences", "households", "places"]:
    con.sql(f"create view {t} as select * from read_parquet('{config.PROCESSED_DIR / (t + '.parquet')}')")
con.sql(open(config.PIPELINE_DIR / "schema.sql").read())
def q(sql): return con.sql(sql).df()
WEB_DATA = config.PROJECT_DIR / "web" / "data"
# DuckDB 2.0: `//` on DOUBLE is float division. Dates and pages are whole numbers stored as DOUBLE: check, then cast to bigint.
assert q("select count(*) n from residences where date <> floor(date) or pageNumber <> floor(pageNumber)").n[0] == 0
q("select (select count(*) from residences) residences, (select count(*) from households) households, (select count(*) from places) places")

,residences,households,places
0,111317,78745,1919


## 0.1 Stops
One row per NAID × person × stop. Residence rows keep their A05 order (`born` → `enlisted` → interim → `now_resides`); the veteran's `served_state` (not `Cont`, not null) goes in as a `served` stop right after `enlisted`, with no date or page. Only S, W, R and BLWT files; the other file types (roll headers, blank numbers, Not Rev, Old War …) are dropped. The 9,350 `most_frequent` fallback rows stay in, as `person = unknown`, `method = most_frequent`.

In [2]:
CLAIMS = "('S', 'W', 'R', 'BLWT')"
RANK = "case kind when 'born' then 0 when 'enlisted' then 1 when 'served' then 2 when 'now_resides' then 4 else 3 end"
con.sql(f'''
create or replace table stops as
with s as (
  select r.NAID, r.person, r.seq as src_seq, r.place_id, r.state, r.precision, r.kind,
         r.date::bigint as date, r.pageNumber::bigint as page, r.method
  from residences r join households h using (NAID) where h.claim_type in {CLAIMS}
  union all
  select NAID, 'veteran', null, served_state, served_state, 'state', 'served', null, null, 'served_state'
  from households where claim_type in {CLAIMS} and served_state is not null and served_state <> 'Cont')
select NAID, person, row_number() over (partition by NAID, person order by {RANK}, src_seq) as seq,
       place_id, state, precision, kind, date, page, method
from s''')
q("select h.claim_type, count(*) residence_rows, count(distinct NAID) files from residences r join households h using (NAID) group by 1 order by 2 desc")

,claim_type,residence_rows,files
0,S,49654,32669
1,W,43909,20871
2,R,15846,9102
3,BLWT,1540,1399
4,number_blank,185,121
5,other,71,61
6,not_rev,60,48
7,old_war,27,24
8,roll_header,13,8
9,single_document,12,12


In [3]:
q("select kind, count(*) stops, count(date) dated, count(*) filter (where precision = 'county') county_level from stops group by 1 order by 2 desc")

,kind,stops,dated,county_level
0,now_resides,74622,57791,31252
1,served,70853,0,0
2,born,14208,10847,6455
3,removed_to,10909,2435,4703
4,married,6170,2728,2468
5,enlisted,4768,832,2057
6,lived_since,272,29,84


## 0.2 Legs
A leg is two consecutive stops of one person where the place changes: compared by county when both ends are counties, else by state. So `cts_hartford → CT` is not a leg, and `nys_tioga → nys_albany` is (a within-state leg). `move_after` / `move_before` are the two stops' dates (a range, often open).
Act cohort from `households.application_year`: 1818–19, 1820–31, 1832–35, 1836–47, 1848+; `unknown` when the file has no application year.

**Decade (T-013, D-025):** the leg's end year if the file states one (the to-stop's `date`: a dated "removed to", "now resides", marriage …), `decade_source = dated`; else the file's `application_year`, `decade_source = application`. Floor to 10. Years before 1770 → 1770; years after 1849 → 1840 with `late = true` (counted on their own). Legs with neither year keep `decade = null`, `decade_source = unknown`, so the totals stay whole.
Caveat: an undated to-stop takes the application year, so e.g. an undated `born → served` or `removed_to` leg lands in the 1820s–1840s, when it was claimed, not when it happened. That is the 1818 / 1832 Act spike D-025 accepts.


In [4]:
COHORT = '''case when application_year is null then 'unknown'
  when application_year <= 1819 then '1818-19' when application_year <= 1831 then '1820-31'
  when application_year <= 1835 then '1832-35' when application_year <= 1847 then '1836-47' else '1848+' end'''
con.sql(f'''
create or replace table people as
select distinct s.NAID, s.person, h.claim_type, h.veteran_name, h.application_year, {COHORT} as act_cohort
from stops s join households h using (NAID)''')
con.sql('''
create or replace table legs as
with s as (
  select *, lag(seq) over w as from_seq, lag(place_id) over w as from_place_id, lag(state) over w as from_state,
         lag(precision) over w as from_precision, lag(kind) over w as from_kind, lag(date) over w as from_date
  from stops window w as (partition by NAID, person order by seq)),
l as (
  select s.NAID, s.person, from_seq, seq as to_seq, from_place_id, place_id as to_place_id, from_state, state as to_state,
         (from_precision = 'county' and precision = 'county') as county_level, from_kind, kind as to_kind,
         from_date as move_after, date as move_before, p.claim_type, p.act_cohort,
         coalesce(date, p.application_year) as decade_year,
         case when date is not null then 'dated' when p.application_year is not null then 'application' else 'unknown' end as decade_source
  from s join people p using (NAID, person)
  where from_seq is not null
    and case when from_precision = 'county' and precision = 'county' then from_place_id <> place_id else from_state <> state end)
select *, case when decade_year is null then null when decade_year < 1770 then 1770 when decade_year > 1849 then 1840
               else decade_year // 10 * 10 end as decade,
       coalesce(decade_year > 1849, false) as late
from l''')
q('''select count(*) legs, count(*) filter (where county_level) county_level, count(*) filter (where from_state = to_state) within_state,
       count(*) filter (where from_kind = 'served' or to_kind = 'served') touch_served,
       count(distinct (NAID, person)) people, count(distinct NAID) files from legs''')

,legs,county_level,within_state,touch_served,people,files
0,39229,3324,1485,27547,30642,28945


**Check against the `moves` view** (`schema.sql`): with the `served` stops left out and only state changes counted, the legs must equal the `moves` rows of S, W, R and BLWT files.

In [5]:
chk = q(f'''
with s as (select * from stops where kind <> 'served'),
l as (select state, lag(state) over (partition by NAID, person order by seq) as fs from s)
select (select count(*) from l where fs is not null and fs <> state) as legs_no_served,
       (select count(*) from moves m join households h using (NAID) where h.claim_type in {CLAIMS}) as moves_view''')
assert chk.legs_no_served[0] == chk.moves_view[0], chk
chk

,legs_no_served,moves_view
0,18868,18868


**Moves per decade** (T-013): legs by decade and by where the year comes from. `late` = year after 1849, inside 1840; `before_1770` = year before 1770, inside 1770; `none` = no year at all.

In [6]:
# moves per decade, split by where the year comes from; late = year after 1849, folded into 1840
dec = q('''select coalesce(decade::varchar, 'none') as decade, count(*) as legs,
       count(*) filter (where decade_source = 'dated') dated, count(*) filter (where decade_source = 'application') application,
       count(*) filter (where decade_source = 'unknown') unknown, count(*) filter (where late) late,
       count(*) filter (where decade_year < 1770) before_1770, count(distinct (NAID, person)) people
from legs group by 1 order by 1''')
print(dec.to_string(index=False))
print("total", dec.legs.sum(), "· dated", dec.dated.sum(), "· application", dec.application.sum(), "· no year", dec.unknown.sum())

decade  legs  dated  application  unknown  late  before_1770  people
  1770   368    368            0        0     0           98     355
  1780   416    416            0        0     0            0     407
  1790   246    246            0        0     0            0     240
  1800   267    267            0        0     0            0     259
  1810  2068   1954          114        0     0            0    2043
  1820  5189   4593          596        0     0            0    5111
  1830 22392  13056         9336        0     0            0   17667
  1840  7769   3160         4609        0  3076            0    6363
  none   514      0            0      514     0            0     462
total 39229 · dated 24060 · application 14655 · no year 514


In [7]:
# where the dated years come from (the to-stop kind), and the cases the rule handles oddly
q('''select to_kind, decade_source, count(*) legs, min(decade_year) min_year, max(decade_year) max_year
from legs group by all order by to_kind, decade_source''')

,to_kind,decade_source,legs,min_year,max_year
0,born,application,76,1820,1863
1,born,dated,81,1747,1841
2,enlisted,application,1058,1832,1868
3,enlisted,dated,279,1736,1879
4,enlisted,unknown,5,<NA>,<NA>
5,lived_since,application,102,1818,1860
6,lived_since,dated,12,1776,1827
7,lived_since,unknown,1,<NA>,<NA>
8,married,application,540,1818,1863
9,married,dated,294,1739,1896


## 0.3 Flows and nodes
**`flows`**: one row per from × to × decade. `level = state` rows cover every leg (within-state legs show as `from = to`); `level = county` rows repeat the legs where both ends are counties, for the click-a-state drill-down. `legs` = leg count, `people` = distinct NAID × person, `served_legs` = legs that start or end at a `served` stop (drop them to see residences only), `dated_legs` = legs whose decade comes from a stated year (the rest use the application year), `late_legs` = legs dated after 1849 (inside 1840), `claim_types` and `act_cohorts` = legs per claim type and per Act cohort. The cohort is a tally, not part of the grain, so the rows don't multiply.

**`nodes`**: one row per place × decade. State nodes count people with a leg out of / into the state from / to another state (within-state legs excluded). County nodes count people with a leg out of / into the county (any leg with a county at that end). `lat/lon` = the plain centroid from `places` (state = mean of county centroids). State nodes add `hub_lat/hub_lon`: the mean of the state's county centroids weighted by people with a residence stop there (all kinds except `served`, all decades; a person counts once per county), and `hub_people` = that weight; a state with no county-level residences keeps its centroid (`hub_people = 0`). A person who moves in two decades counts in both, so per-decade people don't add up to the pooled totals (legs do).


In [8]:
def flows_at(level, a, b, where):
    tally = lambda col: f'''select "from", "to", decade, map_from_entries(list((k, n) order by k)) as counts
        from (select "from", "to", decade, {col} as k, count(*) n from f group by all) group by all'''
    return q(f'''
    with f as (select {a} as "from", {b} as "to", * from legs where {where}),
    m as (select "from", "to", from_state, to_state, decade, count(*) as legs, count(distinct (NAID, person)) as people,
                 count(*) filter (where from_kind = 'served' or to_kind = 'served') as served_legs,
                 count(*) filter (where decade_source = 'dated') as dated_legs, count(*) filter (where late) as late_legs
          from f group by all),
    c as ({tally("claim_type")}), k as ({tally("act_cohort")})
    select '{level}' as level, m.*, c.counts as claim_types, k.counts as act_cohorts
    from m join c on c."from" = m."from" and c."to" = m."to" and c.decade is not distinct from m.decade
           join k on k."from" = m."from" and k."to" = m."to" and k.decade is not distinct from m.decade''')
flows = pd.concat([flows_at("state", "from_state", "to_state", "true"),
                   flows_at("county", "from_place_id", "to_place_id", "county_level")], ignore_index=True)
flows = flows.sort_values(["level", "from", "to", "decade"], na_position="last", ignore_index=True)
flows.groupby("level").agg(rows=("legs", "size"), legs=("legs", "sum"), people=("people", "sum"), dated=("dated_legs", "sum"))

,rows,legs,people,dated
level,,,,
county,3080,3324,3323,1767
state,1869,39229,38911,24060


In [9]:
flows[flows.level == "state"].groupby(["from", "to"]).legs.sum().sort_values(ascending=False).head(12)

from  to
CT    NY    2991
MA    ME    2231
      NY    2012
VA    KY    1581
MA    VT    1522
      NH    1024
CT    VT     977
VA    NC     892
NC    TN     892
VA    TN     787
CT    MA     719
PA    OH     658
Name: legs, dtype: int64

In [10]:
con.sql('''
create or replace table hubs as
with c as (select p.state, s.place_id, any_value(p.lat) lat, any_value(p.lon) lon, count(distinct (s.NAID, s.person)) people
           from stops s join places p using (place_id)
           where s.kind <> 'served' and p.county is not null group by all)
select state as place_id, sum(lat * people) / sum(people) as hub_lat, sum(lon * people) / sum(people) as hub_lon,
       sum(people)::bigint as hub_people, count(*) as hub_counties
from c group by 1''')
nodes = q('''
with e as (
  select 'state' as level, from_state as place_id, decade, NAID, person, 'left' as dir from legs where from_state <> to_state
  union all select 'state', to_state, decade, NAID, person, 'arrived' from legs where from_state <> to_state
  union all select 'county', from_place_id, decade, NAID, person, 'left' from legs l join places p on p.place_id = l.from_place_id where p.county is not null
  union all select 'county', to_place_id, decade, NAID, person, 'arrived' from legs l join places p on p.place_id = l.to_place_id where p.county is not null),
n as (
  select e.level, e.place_id, p.state, p.county, p.lat, p.lon, e.decade,
         count(distinct (NAID, person)) filter (where dir = 'left') as people_left,
         count(distinct (NAID, person)) filter (where dir = 'arrived') as people_arrived
  from e join places p using (place_id) group by all)
select n.*,
       case when n.level = 'state' then coalesce(h.hub_lat, n.lat) end as hub_lat,
       case when n.level = 'state' then coalesce(h.hub_lon, n.lon) end as hub_lon,
       case when n.level = 'state' then coalesce(h.hub_people, 0) end as hub_people
from n left join hubs h on n.level = 'state' and h.place_id = n.place_id
order by level, place_id, decade nulls last''')
nodes.groupby("level").agg(rows=("place_id", "size"), places=("place_id", "nunique"), left=("people_left", "sum"), arrived=("people_arrived", "sum"))

,rows,places,left,arrived
level,,,,
county,3140,1086,6346,16521
state,236,33,37178,37215


In [11]:
# hub vs. plain centroid per state: how far the people-weighted centre moves (km, rough: 111 km per degree)
import numpy as np
hb = (nodes[nodes.level == "state"].drop_duplicates("place_id")[["place_id", "lat", "lon", "hub_lat", "hub_lon", "hub_people"]]
      .merge(q("select place_id, hub_counties from hubs"), on="place_id", how="left"))
hb["shift_km"] = (111 * np.hypot(hb.hub_lat - hb.lat, (hb.hub_lon - hb.lon) * np.cos(np.radians(hb.lat)))).round(0)
print(hb.sort_values("hub_people", ascending=False).round(3).to_string(index=False))

        place_id    lat      lon  hub_lat  hub_lon  hub_people  hub_counties  shift_km
              NY 42.651  -75.410   42.582  -75.228        7448          59.0      17.0
              VA 38.039  -79.168   37.892  -78.608        5171         124.0      52.0
              MA 42.112  -71.438   42.292  -71.654        3818          14.0      27.0
              CT 41.570  -73.799   41.590  -72.698        3500           8.0      91.0
              PA 40.819  -77.683   40.537  -77.172        3227          59.0      53.0
              NC 35.639  -79.292   35.720  -79.549        2706          78.0      25.0
              VT 44.065  -72.658   43.813  -72.750        2378          15.0      29.0
              KY 37.634  -85.344   37.789  -84.989        2241          95.0      36.0
              ME 44.841  -69.454   44.312  -70.011        2085          13.0      73.0
              OH 40.315  -82.856   40.127  -82.547        1951          80.0      33.0
              NH 43.531  -71.580   43.362  

## 0.4 Counts
Legs per kind pair (from kind → to kind), all levels.

In [12]:
q("select from_kind, to_kind, count(*) legs from legs group by all order by legs desc").pivot_table(
    index="from_kind", columns="to_kind", values="legs", fill_value=0, margins=True, aggfunc="sum")

to_kind,born,enlisted,lived_since,married,now_resides,removed_to,served,All
from_kind,,,,,,,,
born,157,1241,2,96,349,151,3008,5004
enlisted,0,101,0,45,256,62,958,1422
lived_since,0,0,2,6,55,5,0,68
married,0,0,7,148,3041,296,0,3492
removed_to,0,0,25,257,3755,1625,0,5662
served,0,0,79,294,17696,5512,0,23581
All,157,1342,115,846,25152,7651,3966,39229


People per Act cohort: all people with stops, people with ≥2 stops, people with ≥1 leg.

In [13]:
q('''
with n as (select NAID, person, count(*) stops from stops group by all),
l as (select distinct NAID, person from legs)
select p.act_cohort, count(*) people, count(*) filter (where n.stops >= 2) people_2plus_stops,
       count(l.NAID) people_with_legs, count(distinct p.NAID) files
from people p join n using (NAID, person) left join l using (NAID, person) group by 1 order by 1''')

,act_cohort,people,people_2plus_stops,people_with_legs,files
0,1818-19,2586,1311,633,2073
1,1820-31,11895,6990,4257,9962
2,1832-35,24445,19735,11959,21780
3,1836-47,26740,10383,6626,15082
4,1848+,22563,10475,6696,12148
5,unknown,16095,833,471,13986


Files with ≥2 stops: counted per file (all people together) and per person (a chain of their own).

In [14]:
q('''
with f as (select NAID, sum(n) stops, max(n) max_person from
             (select NAID, person, count(*) n from stops group by all) group by 1)
select count(*) files_with_stops, count(*) filter (where stops >= 2) files_2plus_stops,
       count(*) filter (where max_person >= 2) files_person_2plus_stops,
       (select count(distinct NAID) from legs) files_with_legs from f''')

,files_with_stops,files_2plus_stops,files_person_2plus_stops,files_with_legs
0,75031,61377,46683,28945


## 0.5 Assertions and export
`paths.json`: one record per NAID × person with its ordered stops (`seq`, `place_id`, `state`, `precision`, `kind`, `date`, `page`; `method` only on the fallback and `served` stops). Split by state only if the file passes 20 MB: then each person goes to `paths/<state>.json` by the state of their last stop, and `paths.json` is the index of parts.

In [15]:
pl_ids = set(q("select place_id from places").place_id)
n_legs, n_cty = q("select count(*) a, count(*) filter (where county_level) b from legs").iloc[0]
DECADES = set(range(1770, 1850, 10))
# unique keys, no null keys
assert q("select count(*) n from (select NAID, person, seq from stops group by all having count(*) > 1)").n[0] == 0
assert q("select count(*) n from stops where NAID is null or person is null or seq is null or place_id is null or kind is null").n[0] == 0
assert q("select count(*) n from (select NAID, person, from_seq from legs group by all having count(*) > 1)").n[0] == 0
assert q("select count(*) n from (select NAID, person from people group by all having count(*) > 1)").n[0] == 0
assert not flows.duplicated(["level", "from", "to", "decade"]).any()
assert not nodes.duplicated(["level", "place_id", "decade"]).any()
# every place_id is in places
assert set(q("select distinct place_id from stops").place_id) <= pl_ids
assert set(flows["from"]) | set(flows["to"]) <= pl_ids and set(nodes.place_id) <= pl_ids
assert nodes[["lat", "lon"]].notna().all().all()
# decades: 1770 ... 1840 or null; null only when the leg has no year at all; late legs sit in 1840
assert set(flows.decade.dropna().astype(int)) <= DECADES and set(nodes.decade.dropna().astype(int)) <= DECADES
assert q("select count(*) n from legs where (decade is null) <> (decade_source = 'unknown')").n[0] == 0
assert q("select count(*) n from legs where late and decade <> 1840").n[0] == 0
assert set(q("select distinct decade_source from legs").decade_source) <= {"dated", "application", "unknown"}
# leg totals = sum of flows, and = the T-005 export (2026-11-23): 39,229 legs, 3,324 county-level, 27,547 served
assert flows[flows.level == "state"].legs.sum() == n_legs == 39229
assert flows[flows.level == "county"].legs.sum() == n_cty == 3324
assert flows[flows.level == "state"].served_legs.sum() == 27547
assert flows.dated_legs.sum() == q("select count(*) n from legs where decade_source = 'dated'").n[0] + \
       q("select count(*) n from legs where decade_source = 'dated' and county_level").n[0]
assert (flows.claim_types.map(lambda d: sum(d.values())) == flows.legs).all()
assert (flows.act_cohorts.map(lambda d: sum(d.values())) == flows.legs).all()
assert (flows.people <= flows.legs).all() and (flows.served_legs <= flows.legs).all()
assert (flows.dated_legs <= flows.legs).all() and (flows.late_legs <= flows.legs).all()
# people: decades split a person who moved twice on one route, so compare distinct people with decades pooled to T-005
pooled = q('''select level, sum(n) people from (
  select 'state' level, count(distinct (NAID, person)) n from legs group by from_state, to_state
  union all select 'county', count(distinct (NAID, person)) from legs where county_level group by from_place_id, to_place_id) group by 1''')
assert dict(zip(pooled.level, pooled.people)) == {"state": 38683, "county": 3320}, pooled
pooled_n = q('''with e as (
  select 'state' lv, from_state p, NAID, person, 'l' d from legs where from_state <> to_state
  union all select 'state', to_state, NAID, person, 'a' from legs where from_state <> to_state
  union all select 'county', from_place_id, NAID, person, 'l' from legs l join places x on x.place_id = l.from_place_id where x.county is not null
  union all select 'county', to_place_id, NAID, person, 'a' from legs l join places x on x.place_id = l.to_place_id where x.county is not null)
select lv, d, sum(n) people from (select lv, d, p, count(distinct (NAID, person)) n from e group by all) group by all''')
assert {(r.lv, r.d): int(r.people) for r in pooled_n.itertuples()} == {
    ("state", "l"): 36838, ("state", "a"): 36886, ("county", "l"): 6329, ("county", "a"): 16467}, pooled_n
# hubs: every state node has one, county nodes don't
st_n = nodes[nodes.level == "state"]
assert st_n[["hub_lat", "hub_lon"]].notna().all().all() and nodes[nodes.level == "county"].hub_lat.isna().all()
# only the four claim types, cohorts as defined
assert set(q("select distinct claim_type from people").claim_type) <= {"S", "W", "R", "BLWT"}
assert set(k for d in flows.act_cohorts for k in d) <= {"1818-19", "1820-31", "1832-35", "1836-47", "1848+", "unknown"}
print("ok:", int(n_legs), "legs ·", int(n_cty), "county-level ·", len(flows), "flow rows ·", len(nodes), "node rows")

ok: 39229 legs · 3324 county-level · 4949 flow rows · 3376 node rows


**Hubs inside their state.** Each state hub is tested against every Newberry AHCB version of that state or territory (`FL` = FL, FL Terr, FL UFT …), 1783–2000. A hub can fall outside when the state's people sit on both sides of a bend or a bay: the mean of a crescent is not on it.

In [16]:
# every state hub inside its state: tested against every Newberry AHCB version of that state or territory (read-only from data/raw)
import geopandas as gpd
from shapely.geometry import Point
ahcb = gpd.read_file(config.RAW_DIR / "gis" / "newberry-ahcb-stateterr" / "shp" / "US_HistStateTerr_Gen001.shp")
ahcb["code"] = ahcb.ABBR_NAME.str.split().str[0]  # 'FL Terr' -> FL, 'OR Cntry' -> OR
CODE = {"California": "CA", "Oregon Territory": "OR"}
def inside(r):
    shapes = ahcb[ahcb.code == CODE.get(r.place_id, r.place_id)]
    return bool(len(shapes)) and bool(shapes.geometry.contains(Point(r.hub_lon, r.hub_lat)).any())
hb["hub_inside"] = hb.apply(inside, axis=1)
hub_pts = gpd.GeoSeries(gpd.points_from_xy(hb.hub_lon, hb.hub_lat), crs=4326).to_crs(5070)
ahcb_m = ahcb.to_crs(5070)  # equal-area metres, for the distance to the state's nearest edge
hb["outside_km"] = [0.0 if ok else round(ahcb_m[ahcb_m.code == CODE.get(pid, pid)].distance(pt).min() / 1000, 1)
                    for pid, ok, pt in zip(hb.place_id, hb.hub_inside, hub_pts)]
outside = hb[~hb.hub_inside]
print(len(hb), "state hubs ·", int(hb.hub_inside.sum()), "inside their state ·", len(outside), "outside:")
print(outside[["place_id", "hub_lat", "hub_lon", "hub_people", "hub_counties", "outside_km"]].to_string(index=False) if len(outside) else "none")
assert hb.hub_lat.notna().all() and hb.hub_lon.notna().all()
assert (hb.outside_km < 10).all(), "a hub sits more than 10 km outside its state"

33 state hubs · 31 inside their state · 2 outside:
place_id   hub_lat    hub_lon  hub_people  hub_counties  outside_km
      FL 30.074300 -84.133118          22          13.0         0.9
      LA 31.206676 -91.503465          17          11.0         7.5


In [17]:
def clean(v):
    if v is pd.NA or (isinstance(v, float) and v != v): return None
    if hasattr(v, "item"): return v.item()
    return v
def records(df): return [{k: clean(v) for k, v in r.items()} for r in df.to_dict("records")]

st = q("select * from stops order by NAID, person, seq")
st["method"] = st.method.where(st.method.isin(["most_frequent", "served_state"]))
STOP_COLS = ["seq", "place_id", "state", "precision", "kind", "date", "page", "method"]
by_person = {k: [{c: v for c, v in s.items() if v is not None} for s in records(g[STOP_COLS])]
             for k, g in st.groupby(["NAID", "person"], sort=False)}
ppl = q("select * from people order by NAID, person")
paths = [{"NAID": r["NAID"], "person": r["person"], "name": r["veteran_name"], "claim_type": r["claim_type"],
          "application_year": r["application_year"], "act_cohort": r["act_cohort"], "stops": by_person[(r["NAID"], r["person"])]}
         for r in records(ppl)]

WEB_DATA.mkdir(parents=True, exist_ok=True)
def dump(obj, name):
    p = WEB_DATA / name
    p.write_text(json.dumps(obj, separators=(",", ":"), allow_nan=False))
    return p, round(p.stat().st_size / 1e6, 2)
out = [dump(records(flows), "flows.json"), dump(records(nodes), "nodes.json")]
size = len(json.dumps(paths, separators=(",", ":"), allow_nan=False).encode()) / 1e6
print("paths as one file:", round(size, 1), "MB")
if size <= 20:
    out.append(dump(paths, "paths.json"))
else:
    # split by the state of each person's last stop (where the file was filed from); paths.json becomes the index
    (WEB_DATA / "paths").mkdir(exist_ok=True)
    parts = {}
    for r in paths: parts.setdefault(r["stops"][-1]["state"], []).append(r)
    index = []
    for stt, rs in sorted(parts.items()):
        fn = "paths/" + stt.replace(" ", "_") + ".json"
        _, mb = dump(rs, fn)
        index.append({"state": stt, "file": fn, "people": len(rs), "stops": sum(len(r["stops"]) for r in rs), "mb": mb})
    out.append(dump({"split_by": "state of last stop", "parts": index}, "paths.json"))
    print(pd.DataFrame(index).sort_values("mb", ascending=False).head(8).to_string(index=False))
out

paths as one file: 32.5 MB


state          file  people  stops   mb
   NY paths/NY.json   17593  33275 5.74
   MA paths/MA.json   14556  20766 4.07
   VA paths/VA.json    9011  14117 2.65
   CT paths/CT.json    9225  13579 2.63
   PA paths/PA.json    7027  11495 2.12
   VT paths/VT.json    4803   9980 1.65
   NH paths/NH.json    5430   8480 1.60
   NC paths/NC.json    5317   8606 1.59


[(PosixPath('/Users/larawashington/Documents/Parsons/MajorStudio1/Projects/revolutionary-xroads/web/data/flows.json'),
  1.1),
 (PosixPath('/Users/larawashington/Documents/Parsons/MajorStudio1/Projects/revolutionary-xroads/web/data/nodes.json'),
  0.69),
 (PosixPath('/Users/larawashington/Documents/Parsons/MajorStudio1/Projects/revolutionary-xroads/web/data/paths.json'),
  0.0)]

In [18]:
# round trip: the files on disk hold what the tables hold
f = json.loads((WEB_DATA / "flows.json").read_text()); n = json.loads((WEB_DATA / "nodes.json").read_text())
p = json.loads((WEB_DATA / "paths.json").read_text())
if "parts" in p:
    p = [r for part in p["parts"] for r in json.loads((WEB_DATA / part["file"]).read_text())]
assert len(p) == len(ppl) and sum(len(r["stops"]) for r in p) == len(st)
assert len({(r["NAID"], r["person"]) for r in p}) == len(p)
assert all([s["seq"] for s in r["stops"]] == list(range(1, len(r["stops"]) + 1)) for r in p)
assert len(f) == len(flows) and len(n) == len(nodes)
assert sum(r["legs"] for r in f if r["level"] == "state") == 39229 and sum(r["legs"] for r in f if r["level"] == "county") == 3324
assert {r["decade"] for r in f} <= DECADES | {None} and all(isinstance(r["decade"], int) for r in f if r["decade"] is not None)
assert all(r["hub_lat"] is not None for r in n if r["level"] == "state")
sizes = {fn: round((WEB_DATA / fn).stat().st_size / 1e6, 2) for fn in ["flows.json", "nodes.json"]}
assert all(mb < 3 for mb in sizes.values()), sizes
print(sizes)
print(len(f), "flows ·", len(n), "nodes ·", len(p), "people ·", len(st), "stops")

{'flows.json': 1.1, 'nodes.json': 0.69}
4949 flows · 3376 nodes · 104324 people · 181802 stops


## Counts (run 2026-11-23)
- Stops: 181,802 for 104,324 people in 75,031 files (S, W, R, BLWT); 70,853 are `served` stops. 61,377 files have ≥2 stops; 46,683 have one person with ≥2 stops.
- Legs: **39,229** for 30,642 people in 28,945 files; 3,324 county-level; 1,485 within one state; 27,547 (70%) start or end at a `served` stop. Without `served`: 18,868 legs = the `moves` view for these claim types.
- Largest kind pairs: served → now_resides 17,696 · served → removed_to 5,512 · removed_to → now_resides 3,755 · married → now_resides 3,041 · born → served 3,008.
- People with legs by cohort: 1818–19 633 · 1820–31 4,257 · 1832–35 11,959 · 1836–47 6,626 · 1848+ 6,696 · unknown 471.
- Files: `flows.json` 4,892 rows (1,778 state, 3,114 county), `nodes.json` 3,343 rows (177 state, 3,166 county), paths 32.5 MB as one file, so split into 33 files in `paths/` by the state of the last stop (largest NY 5.7 MB); `paths.json` is the index.

## Counts (T-013 re-run, 2026-11-24)
- Decades (all 39,229 legs, 24,060 dated by a stated year, 14,655 by the application year, 514 with no year → `decade = null`): 1770 368 · 1780 416 · 1790 246 · 1800 267 · 1810 2,068 (114 application) · 1820 5,189 (596) · 1830 22,392 (9,336) · 1840 7,769 (4,609; 3,076 of them `late`, dated after 1849). 98 legs dated before 1770 sit in 1770.
- Totals unchanged from T-005: 39,229 legs, 3,324 county-level, 27,547 served; distinct people per route and per node with decades pooled match the T-005 export (state flows 38,683; state nodes 36,838 left / 36,886 arrived; county nodes 6,329 / 16,467).
- Files: `flows.json` 4,949 rows (1,869 state, 3,080 county), 1.1 MB; `nodes.json` 3,376 rows (236 state, 3,140 county), 0.69 MB. `paths/` unchanged.
- Hubs: 33 state nodes; 32 weighted by people (DC has no county residences and keeps its centroid). 31 hubs fall inside their state; FL is 0.9 km offshore in Apalachee Bay (generalised coastline) and LA is 7.5 km over the Mississippi into MS (its 17 people sit on both sides of the river). Largest shifts from the centroid: Oregon Territory 184 km and California 177 km (1–3 people), WI 116, SC 115, MI 107, CT 91 km.
